# Module 01: NumPy for Machine Learning
## Notebook 04: Mathematics, Statistics, Linear Algebra, and PCA from Scratch

Machine learning models are mathematical systems expressed through linear algebra, vector calculus, and probability. In this notebook, we move from fundamental statistical reductions to advanced matrix factorizations (SVD, Cholesky) and implement Principal Component Analysis (PCA) entirely from scratch.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Compute axis-wise statistical reductions and utilize `keepdims=True` for robust broadcasting.
2. Execute vector dot products and matrix multiplications using the `@` operator.
3. Solve linear equations, compute matrix inverses, and evaluate determinants.
4. Compute $L_1$, $L_2$, and Frobenius norms for regularization penalties (Lasso / Ridge).
5. Generate reproducible distributions using modern `np.random.default_rng`.
6. Factorize matrices using **Singular Value Decomposition (SVD)** for low-rank compression.
7. Solve normal equations stably using **Cholesky Decomposition**.
8. Implement an end-to-end **Principal Component Analysis (PCA)** algorithm from scratch.

In [ ]:
import numpy as np

print(f"NumPy version: {np.__version__}")

### 1. Statistical Reductions Along Axes & `keepdims`

When working with a 2D data matrix $X$ of shape $(N_{samples}, N_{features})$:
- `axis=0`: Collapses rows (computes statistics **per feature** across all samples).
- `axis=1`: Collapses columns (computes statistics **per sample** across all features).

#### Why `keepdims=True` is Essential:
Without `keepdims=True`, reducing along an axis drops that dimension (e.g., shape `(5, 3)` becomes `(3,)`).
With `keepdims=True`, the reduced dimension is kept as length 1 (shape `(1, 3)` or `(5, 1)`), making broadcasted operations guaranteed to align properly!

In [ ]:
# 5 samples, 3 features
X = np.array([
    [10.0, 25.0, 5.0],
    [12.0, 30.0, 8.0],
    [9.0,  20.0, 6.0],
    [15.0, 40.0, 12.0],
    [14.0, 35.0, 9.0]
])

# Feature-wise statistics (axis=0)
mean_feat = np.mean(X, axis=0, keepdims=True)
std_feat = np.std(X, axis=0, keepdims=True)
max_feat = np.max(X, axis=0, keepdims=True)

print(f"Original shape:        {X.shape}")
print(f"Mean shape (keepdims): {mean_feat.shape}")
print("Feature Means:         ", mean_feat)
print("Feature Std Devs:      ", std_feat)

# Guaranteed broadcast: (5, 3) - (1, 3)
X_norm = (X - mean_feat) / std_feat
print("\nNormalized Data:\n", np.round(X_norm, 3))

---
### 2. Matrix Multiplication and Dot Products

> **Crucial Rule:**
> - `*` performs **element-wise** multiplication (Hadamard product).
> - `@` (or `np.matmul()`) performs **true matrix multiplication** ($C_{ik} = \sum_j A_{ij} B_{jk}$).

In [ ]:
# Vector Dot Product: u . v = \sum u_i * v_i
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])

dot_product = np.dot(u, v) # or u @ v
print(f"Dot product (1*4 + 2*5 + 3*6): {dot_product}")

# Matrix-Vector Multiplication: y_pred = X @ w
# 3 samples, 2 features
X_mat = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0]
])
# Weight vector: 2 features -> 1 output
w = np.array([0.5, -0.2])

predictions = X_mat @ w
print("\nFeature Matrix X (3x2):\n", X_mat)
print("Weights w (2,):", w)
print("Predictions X @ w (3,):", predictions)

# Matrix Transposition
print("\nTransposed X (2x3):\n", X_mat.T)

---
### 3. Linear Algebra Submodule: `np.linalg`

NumPy's `np.linalg` wraps optimized BLAS and LAPACK routines.

#### Key Functions:
- `np.linalg.det(A)`: Computes matrix determinant.
- `np.linalg.inv(A)`: Computes matrix inverse $A^{-1}$ ($A A^{-1} = I$).
- `np.linalg.pinv(A)`: Computes Moore-Penrose pseudo-inverse (works even if matrix is singular or non-square!).
- `np.linalg.solve(A, b)`: Solves linear system $A x = b$ (more stable and faster than `inv(A) @ b`).

In [ ]:
# Solving a linear system:
# 2x + 3y = 8
# 4x + 9y = 20
A = np.array([[2.0, 3.0], [4.0, 9.0]])
b = np.array([8.0, 20.0])

det_A = np.linalg.det(A)
print(f"Determinant of A: {det_A:.2f}")

# Solve Ax = b
solution = np.linalg.solve(A, b)
print(f"Solution [x, y]:   {solution}")

# Verify solution: A @ solution == b
print("Verification A @ x:", A @ solution)

---
### 4. Vector and Matrix Norms (Regularization)

Norms quantify the "magnitude" of vectors and matrices, forming the mathematical backbone of regularization:
- **$L_1$ Norm (Manhattan / Lasso penalty)**: $\|w\|_1 = \sum |w_i|$ (encourages sparsity/feature selection).
- **$L_2$ Norm (Euclidean / Ridge penalty)**: $\|w\|_2 = \sqrt{\sum w_i^2}$ (shrinks weights toward zero).
- **Frobenius Norm**: Matrix equivalent of $L_2$ norm.

In [ ]:
w_weights = np.array([3.0, -4.0, 0.0, 1.5, -0.5])

# L1 Norm (Lasso penalty)
l1_norm = np.linalg.norm(w_weights, ord=1)
print(f"L1 Norm (Sum of absolute values): {l1_norm}")

# L2 Norm (Euclidean length / Ridge penalty)
l2_norm = np.linalg.norm(w_weights, ord=2)
print(f"L2 Norm (Euclidean magnitude):    {l2_norm:.4f}")

# Frobenius norm of a weight matrix
W_matrix = np.array([[1.0, 2.0], [3.0, 4.0]])
frobenius_norm = np.linalg.norm(W_matrix, ord='fro')
print(f"Frobenius Norm of matrix:         {frobenius_norm:.4f}")

---
### 5. Modern Random Number Generation (`default_rng`)

> **NumPy Modern Practice:**
> Avoid legacy `np.random.seed()` or `np.random.randn()`.
> Always instantiate a `Generator` via `np.random.default_rng(seed)`.
> It uses the high-performance **PCG64** bit generator, has superior statistical properties, and is thread-safe!

In [ ]:
# 1. Instantiate the generator with a fixed seed for reproducible experiments
rng = np.random.default_rng(seed=42)

# 2. Gaussian / Normal distribution: N(mean=0, std=1)
weights = rng.normal(loc=0.0, scale=0.1, size=(3, 4))
print("Gaussian Weights (3x4):\n", np.round(weights, 4))

# 3. Uniform distribution: U(low=0.0, high=1.0)
uniform_noise = rng.uniform(low=0.0, high=1.0, size=5)
print("\nUniform random values:", np.round(uniform_noise, 4))

# 4. Random permutation for train/test splitting
indices = rng.permutation(10)
print("\nShuffled indices of 10 samples:", indices)

---
### 6. Advanced Usages: SVD, Cholesky Decomposition, and PCA from Scratch

#### A. Singular Value Decomposition (SVD) for Low-Rank Compression

Any real matrix $A$ of shape $(M, N)$ can be decomposed into:
$$A = U \Sigma V^T$$
- $U$: $(M, M)$ orthogonal matrix of left singular vectors.
- $\Sigma$: $(M, N)$ diagonal matrix of singular values in descending order $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$.
- $V^T$: $(N, N)$ orthogonal matrix of right singular vectors.

**Truncated SVD (Rank-$k$ Approximation):**
According to the Eckart-Young-Mirsky Theorem, the optimal rank-$k$ approximation minimizing reconstruction error $\|A - A_k\|_F$ is obtained by keeping only the top $k$ singular values:
$$A_k = U_{:, :k} \cdot \text{diag}(\Sigma_{:k}) \cdot V_{:k, :}^T$$

In [ ]:
# Generate synthetic 20x15 feature matrix
rng = np.random.default_rng(101)
A_matrix = rng.normal(0, 1, size=(20, 15))

# Compute Full SVD
U, s, Vt = np.linalg.svd(A_matrix, full_matrices=False)

print(f"Original A shape: {A_matrix.shape}")
print(f"U shape: {U.shape}, s shape: {s.shape}, Vt shape: {Vt.shape}")
print("Top 5 singular values:", np.round(s[:5], 3))

# Rank-3 Low-Rank Approximation
k = 3
A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

# Measure Frobenius reconstruction error
reconstruction_error = np.linalg.norm(A_matrix - A_k, ord='fro')
total_energy = np.linalg.norm(A_matrix, ord='fro')
variance_preserved = (np.sum(s[:k] ** 2) / np.sum(s ** 2)) * 100

print(f"\nRank-{k} Approximation Frobenius Error: {reconstruction_error:.4f}")
print(f"Energy / Variance preserved by rank-{k}:  {variance_preserved:.2f}%")

#### B. Cholesky Decomposition for Stable Linear Solvers

In Ridge Regression and Gaussian Processes, we solve systems of the form:
$$(X^T X + \lambda I) w = X^T y$$
The matrix $A = (X^T X + \lambda I)$ is guaranteed to be **Symmetric and Positive-Definite (SPD)**.
Computing an explicit matrix inverse $(X^T X + \lambda I)^{-1}$ is computationally wasteful ($O(n^3)$) and numerically unstable.

**Cholesky Factorization:**
$$A = L L^T$$
where $L$ is a lower triangular matrix.
Solving $L L^T w = b$ requires two simple triangular solves (forward and back substitution), which is **twice as fast** as standard LU decomposition and never suffers from catastrophic cancellation!

In [ ]:
def solve_spd_cholesky(A, b):
    # A must be symmetric positive-definite
    # 1. Compute lower-triangular Cholesky factor L: A = L @ L.T
    L = np.linalg.cholesky(A)
    
    # 2. Solve L y = b for y (forward substitution)
    y = np.linalg.solve(L, b)
    
    # 3. Solve L.T w = y for w (backward substitution)
    w = np.linalg.solve(L.T, y)
    return w

# Test with a regularized design matrix
d = 4
X_sample = rng.normal(0, 1, size=(50, d))
A_spd = X_sample.T @ X_sample + 1.0 * np.eye(d)  # Strictly SPD
b_rhs = rng.normal(0, 1, size=d)

# Solve using Cholesky
w_chol = solve_spd_cholesky(A_spd, b_rhs)
# Compare with standard solver
w_direct = np.linalg.solve(A_spd, b_rhs)

print("Cholesky Solution: ", np.round(w_chol, 6))
print("Direct LAPACK Sol: ", np.round(w_direct, 6))
print("Max absolute discrepancy:", np.max(np.abs(w_chol - w_direct)))

#### C. Principal Component Analysis (PCA) End-to-End from Scratch

PCA is the cornerstone of unsupervised dimensionality reduction. Here is the complete linear algebraic pipeline:
1. **Centering**: Subtract feature means: $X_c = X - \mu$.
2. **Covariance Matrix**: $\Sigma = \frac{1}{N - 1} X_c^T X_c$.
3. **Eigendecomposition**: $\Sigma v_i = \lambda_i v_i$ using `np.linalg.eigh`.
4. **Sorting**: Sort eigenvalues in descending order to identify directions of maximal variance.
5. **Projection**: Transform samples into the principal subspace: $Z = X_c W_k$.

In [ ]:
class PCAScratch:
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.mean_ = None
        self.components_ = None
        self.explained_variance_ratio_ = None
        
    def fit(self, X):
        N = X.shape[0]
        # 1. Compute mean and center data
        self.mean_ = np.mean(X, axis=0, keepdims=True)
        X_centered = X - self.mean_
        
        # 2. Compute sample covariance matrix
        cov = (X_centered.T @ X_centered) / (N - 1)
        
        # 3. Eigendecomposition of symmetric covariance matrix
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        
        # 4. Sort eigenvalues and eigenvectors in descending order
        sorted_indices = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]
        
        # 5. Store top components and variance explained
        self.components_ = eigenvectors[:, :self.n_components]
        total_var = np.sum(eigenvalues)
        self.explained_variance_ratio_ = (eigenvalues[:self.n_components] / total_var)
        return self
        
    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_
        
    def inverse_transform(self, Z):
        return (Z @ self.components_.T) + self.mean_

# Test PCA from scratch on 5D synthetic dataset
X_high_dim = rng.normal(loc=[10, 5, -2, 0, 8], scale=[3, 1, 0.5, 0.1, 2], size=(200, 5))

pca = PCAScratch(n_components=2)
pca.fit(X_high_dim)
X_projected = pca.transform(X_high_dim)

print("Original shape:  ", X_high_dim.shape)
print("Projected shape: ", X_projected.shape)
print("Explained Variance Ratio per PC: ", np.round(pca.explained_variance_ratio_, 4))
print(f"Total variance captured by 2 PCs: {np.sum(pca.explained_variance_ratio_) * 100:.2f}%")

### Summary & Next Steps
In this notebook, you mastered:
- Computing axis-wise statistical reductions with `keepdims=True`.
- Vector dot products and matrix multiplications (`@`).
- Matrix determinants, solving linear systems, and inversions.
- Regularization norms ($L_1$, $L_2$, Frobenius).
- Modern random generation with `np.random.default_rng`.
- **Singular Value Decomposition (SVD)** and low-rank matrix approximations.
- **Cholesky Factorization** for stable, fast SPD normal equation solving.
- Implementing **Principal Component Analysis (PCA)** end-to-end from first principles.

**Next Notebook:** `05_practical_ml_applications.ipynb` — Implement Ridge Regression, Mini-Batch SGD with Momentum, and Multi-Class Softmax Regression from scratch!